# IDEB: da fonte oficial ao Power BI

Demonstracao do ciclo do IDEB no recorte analitico 2007-2023. O arquivo Raw canonico pode conter edicao posterior, mas o recorte e controlado pela Silver. A saida para o BI e `data/gold/fatos/fato_ideb.parquet`.

## Ciclo

Raw canonica -> auditorias de notas, redes, UFs e serie -> Bronze por etapa -> Silver 2007-2023 -> fato Gold -> Power BI. A substituicao do workbook Raw nao amplia automaticamente o periodo analitico.

## Criterios de tratamento e linhagem

A Bronze preserva as abas do workbook por etapa. A Silver utiliza `ANOS_INICIAIS` e `ANOS_FINAIS`, converte os nomes geograficos para as 27 siglas de UF e seleciona explicitamente a categoria oficial `Publica (4)`. Brasil, regioes e outras redes nao entram no recorte.

O pipeline procura pelos nomes tecnicos `VL_OBSERVADO_2007` ate `VL_OBSERVADO_2023`; portanto, uma coluna de 2025 existente no arquivo fisico nao entra automaticamente nesta versao. Os nove anos bienais sao transformados de colunas para linhas. O total esperado e `9 edicoes x 27 UFs x 2 etapas = 486`.

`REDE = PUBLICA` e a categoria canonica. `REDE_ORIGEM`, geografia, arquivo, aba, linha e coluna preservam a origem na Silver. A Gold mantem apenas `ANO`, `UF`, `ETAPA`, `REDE` e `IDEB`, sem recalcular o indicador.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
DIRETORIO_ATUAL = Path.cwd().resolve()
ROOT = next((p for p in (DIRETORIO_ATUAL, *DIRETORIO_ATUAL.parents) if (p / 'src' / 'pipeline.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Nao foi possivel localizar a raiz do repositorio.')
print(f'Raiz do projeto: {ROOT}')
EXECUTAR_AUDITORIAS = False
EXECUTAR_PIPELINE_INDICADOR = False
pd.set_option('display.max_columns', 30)

In [ ]:
AUDITORIAS = [
 'src/auditoria_ideb/inspecionar_ideb.py',
 'src/auditoria_ideb/verificar_notas_ideb.py',
 'src/auditoria_ideb/verificar_redes_ideb.py',
 'src/auditoria_ideb/verificar_ufs_redes_ideb.py',
 'src/auditoria_ideb/verificar_ufs_faltantes_ideb.py',
 'src/auditoria_ideb/verificar_serie_publica_ideb.py',
]
ETAPAS = [
 'src/bronze/ideb/ingest_ideb.py',
 'src/bronze/ideb/validar_bronze_ideb.py',
 'src/silver/ideb/transformar_ideb.py',
 'src/silver/ideb/validar_silver_ideb.py',
 'src/gold/ideb/transformar_ideb.py',
 'src/gold/ideb/validar_fato_ideb.py',
]
def executar(scripts):
    for script in scripts:
        print(f'\n>>> {script}')
        subprocess.run([sys.executable, script], cwd=ROOT, check=True)
print('Auditorias:', *AUDITORIAS, sep='\n- ')
print('\nEtapas:', *ETAPAS, sep='\n- ')

In [ ]:
if EXECUTAR_AUDITORIAS:
    executar(AUDITORIAS)
else:
    print('Auditorias nao executadas.')
if EXECUTAR_PIPELINE_INDICADOR:
    executar(ETAPAS)
else:
    print('Pipeline nao executado; leitura dos artefatos existentes.')

## Evidencias por camada

A celula seguinte diferencia o arquivo fisico disponivel na Raw do recorte efetivamente materializado na Silver e na Gold.

In [ ]:
raw = sorted((ROOT / 'data/raw/ideb').glob('*'))
bronze = sorted((ROOT / 'data/bronze/ideb').glob('*.parquet'))
silver_path = ROOT / 'data/silver/ideb/ideb_2007_2023.parquet'
gold_path = ROOT / 'data/gold/fatos/fato_ideb.parquet'
pd.DataFrame({'camada':['Raw','Bronze','Silver','Gold'], 'arquivos':[len(raw),len(bronze),int(silver_path.exists()),int(gold_path.exists())], 'destino':[str(ROOT/'data/raw/ideb'),str(ROOT/'data/bronze/ideb'),str(silver_path),str(gold_path)]})

In [ ]:
amostra_bronze = pd.read_parquet(bronze[0])
silver = pd.read_parquet(silver_path)
gold = pd.read_parquet(gold_path)
display(amostra_bronze.head(3), silver.head(3), gold.head(3))
pd.DataFrame({'camada':['Bronze (uma etapa)','Silver','Gold'], 'linhas':[len(amostra_bronze),len(silver),len(gold)], 'anos':[sorted(amostra_bronze['ANO'].dropna().unique().tolist()) if 'ANO' in amostra_bronze else 'ver esquema', sorted(silver['ANO'].unique().tolist()), sorted(gold['ANO'].unique().tolist())]})

### Evidencia do recorte e da linhagem

Estas tabelas comprovam qual categoria de rede entrou na serie, quais anos foram materializados e quais campos permanecem somente na Silver para auditoria.

In [ ]:
padronizacao_rede = silver.groupby(['REDE_ORIGEM','REDE'], as_index=False).agg(UFS=('UF','nunique'), REGISTROS=('ANO','size'))
display(padronizacao_rede)
display(pd.DataFrame({'ANOS_DISPONIVEIS_NA_GOLD':[', '.join(map(str, sorted(gold['ANO'].unique())))]}))
linhagem = ['ANO','UF','ETAPA','IDEB','REDE','REDE_ORIGEM','GEOGRAFIA_ORIGEM','ARQUIVO_ORIGEM','ABA_ORIGEM','LINHA_ORIGEM_BRONZE','COLUNA_ORIGEM']
display(silver.loc[:, linhagem].head(4))
pd.DataFrame({'Silver_preserva_para_auditoria':[', '.join(c for c in silver.columns if c not in gold.columns)], 'Gold_entrega_ao_BI':[', '.join(gold.columns)]})

In [ ]:
resumo = gold.groupby(['ANO','ETAPA'], as_index=False)['IDEB'].mean()
grafico = resumo.pivot(index='ANO', columns='ETAPA', values='IDEB')
ax = grafico.plot(figsize=(10,5), marker='o')
ax.set(title='IDEB medio das UFs da rede publica', xlabel='Ano da edicao', ylabel='IDEB (0-10)')
ax.grid(alpha=0.25)
plt.show()

## Disponibilizacao para o BI

A fato e relacionada a `DIM_UF`, `DIM_TEMPO` e `DIM_ETAPA`. O workbook Raw corrente e o hash do manifesto identificam a fonte fisica; a Gold acima identifica o recorte 2007-2023 desta versao. Para o modelo integrado, use `python src/pipeline.py full`.